# Anomaly Detection Tutorial

Build a simple baseline over alert volumes and flag outlier hours for analyst review.

In [ ]:
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from clario360 import Client

WINDOW_DAYS = int(__import__('os').environ.get('CLARIO360_ANOMALY_WINDOW_DAYS', '14'))
Z_THRESHOLD = float(__import__('os').environ.get('CLARIO360_Z_THRESHOLD', '2.5'))
sdk = Client.from_env()


## Load alert telemetry

We use alert creation times as a simple anomaly-detection training set.

In [ ]:
alerts = sdk.cyber.alerts.list(date_from=(datetime.utcnow() - timedelta(days=WINDOW_DAYS)).isoformat(), per_page=500)
df = alerts.to_dataframe()
if df.empty:
    raise RuntimeError('No alerts available for anomaly training in the selected time window.')
df['hour'] = pd.to_datetime(df['created_at']).dt.floor('h')
series = df.groupby('hour').size().rename('count').reset_index()
series['zscore'] = (series['count'] - series['count'].mean()) / max(series['count'].std(ddof=0), 1)
series['is_anomaly'] = series['zscore'].abs() >= Z_THRESHOLD
series.tail(10)

## Review anomalies

This highlights unusual alert surges or drops that may correspond to active campaigns, ingestion failures, or tuning regressions.

In [ ]:
colors = series['is_anomaly'].map({True: 'crimson', False: 'steelblue'})
plt.figure(figsize=(12, 4))
plt.bar(series['hour'], series['count'], color=colors)
plt.title('Hourly Alert Volume with Statistical Outliers')
plt.xlabel('Hour')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
series[series['is_anomaly']].sort_values('zscore', ascending=False)
